# Geospatial Streaming Analytics with Stratified Sampling

This notebook demonstrates:
* **Spark Structured Streaming** with tumbling windows (no watermarks)
* **Geohash-based stratified sampling** for spatial data reduction
* **Spatial joins** using geopandas and shapely for point-in-polygon operations
* **Accuracy comparison** between full streaming data and sampled data using RMSE and MAPE

## Configuration
* **Sampling Fraction**: 80% per geohash stratum
* **Geohash Precision**: 6 (approximately 609m × 1200m cells)
* **Window Duration**: 10 seconds (tumbling windows)
* **Dataset**: Chicago air quality sensor data (129,531 records)
* **Neighborhoods**: 98 Chicago neighborhoods from GeoJSON

## Cell 1: Streaming Pipeline with Geohash Stratified Sampling

### What This Cell Does:

**1. Setup & Configuration**
* Defines schema for air quality sensor data (PM2.5, temperature, humidity, etc.)
* Configures sampling parameters and window duration
* Prepares streaming directory and checkpoint locations

**2. Load Geospatial Data**
* Loads Chicago neighborhood boundaries from GeoJSON using geopandas
* Broadcasts neighborhood geometries to all Spark workers for efficient spatial joins

**3. Define UDFs**
* `geohash_udf`: Generates geohash codes (precision 6) for each sensor location
* `find_neighborhood_udf`: Performs point-in-polygon check using shapely to find which neighborhood contains each sensor reading

**4. Streaming Path (Full Data)**
* Reads CSV as a **Spark Structured Stream**
* Adds geohash and neighborhood columns using UDFs
* Aggregates by **10-second tumbling windows** and neighborhood (NO watermark)
* Writes results to in-memory table `full_data_results`
* Polls until streaming query processes data (up to 60 seconds)

**5. Batch Path (Sampled Data)**
* Reads same CSV as **batch** (required for sampling)
* Generates geohash for each record
* Performs **stratified sampling by geohash** (80% from each geohash cell)
* Applies same spatial join and window aggregation as streaming path

**6. Comparison**
* Joins full streaming results with sampled batch results
* Calculates error metrics: squared error and absolute percentage error
* Groups by neighborhood to compute average PM2.5 for full vs sampled data
* Displays detailed comparison and neighborhood-level aggregations

**7. Cleanup**
* Stops the streaming query

### Key Outputs:
* `comparison`: Window-level comparison of full vs sampled aggregations
* `neighborhood_avg`: Neighborhood-level PM2.5 averages for full and sampled data

In [0]:
import os
print(os.environ['PYTHONPATH'])

In [0]:
# Install a pip package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install geopandas geohash2 DBUtils


In [0]:
from pyspark.sql import SparkSession

spark: SparkSession = SparkSession.builder.appName(
    "air_qualy").getOrCreate()

# set log level to WARN
spark.sparkContext.setLogLevel("WARN")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType
from pyspark.sql.functions import col, window, udf
from pyspark.sql import functions as F
import geopandas as gpd
from shapely.geometry import Point
import geohash2
import time
import os
import shutil
import urllib.request

# Configuration
SAMPLING_FRACTION = 0.8  # Sample 80% from each geohash
GEOHASH_PRECISION = 6
WINDOW_DURATION = "10 seconds"  # Tumbling window size

# Define schema
schema = StructType([
    StructField("City", StringType(), True),
    StructField("DeviceId", StringType(), True),
    StructField("LocationName", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("ReadingDateTimeUTC", TimestampType(), True),
    StructField("PM25", DoubleType(), True),
    StructField("CalibratedPM25", DoubleType(), True),
    StructField("CalibratedO3", DoubleType(), True),
    StructField("CalibratedNO2", DoubleType(), True),
    StructField("CO", DoubleType(), True),
    StructField("Temperature", DoubleType(), True),
    StructField("Humidity", DoubleType(), True),
    StructField("BatteryLevel", IntegerType(), True),
    StructField("PercentBattery", DoubleType(), True),
    StructField("CellSignal", StringType(), True)
])

schemaEdge = StructType([
    StructField("City", StringType(), True),
    StructField("DeviceId", StringType(), True),
    StructField("LocationName", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("ReadingDateTimeUTC", TimestampType(), True),
    StructField("PM25", DoubleType(), True),
    StructField("CalibratedPM25", DoubleType(), True),
    StructField("CalibratedO3", DoubleType(), True),
    StructField("CalibratedNO2", DoubleType(), True),
    StructField("CO", DoubleType(), True),
    StructField("Temperature", DoubleType(), True),
    StructField("Humidity", DoubleType(), True),
    StructField("BatteryLevel", IntegerType(), True),
    StructField("PercentBattery", DoubleType(), True),
    StructField("CellSignal", StringType(), True),
    StructField("geohash", StringType(), True),
    StructField("neighborhood", StringType(), True)
])

print(f"Configuration:")
print(f"  Sampling Fraction: {SAMPLING_FRACTION}")
print(f"  Geohash Precision: {GEOHASH_PRECISION}")
print(f"  Tumbling Window: {WINDOW_DURATION}")
print(f"  Watermark: None (disabled)")

start_time = time.time()

# Setup paths - using URLs
csv_url = "https://raw.githubusercontent.com/IsamAljawarneh/datasets/refs/heads/master/data/Chicago/AQ_data/chicago_eclipse_data_part_1.csv"
csv_edge_url = "https://raw.githubusercontent.com/lorenzofelletti/EdgeCloudApproximate/refs/heads/master/out-5/combined-08-new.csv"
geojson_url = "https://raw.githubusercontent.com/blackmad/neighborhoods/refs/heads/master/chicago.geojson"

streaming_dir = "/dbfs/tmp/chicago_streaming_input"
checkpoint_path = "/dbfs/tmp/chicago_stream_checkpoint"

# Clean up and prepare
if os.path.exists(checkpoint_path):
    shutil.rmtree(checkpoint_path)
if os.path.exists(streaming_dir):
    shutil.rmtree(streaming_dir)
'''
os.mkdir(streaming_dir)
stream_data_file = f"{streaming_dir}/data.csv"

# Download CSV from URL
print(f"Downloading CSV from {csv_url}...")
urllib.request.urlretrieve(csv_url, stream_data_file)
print(f"CSV downloaded to {stream_data_file}")
'''
# Clean up and prepare (edge)
streaming_e_dir = "/dbfs/tmp/chicago_streaming_e_input"
checkpoint_e_path = "/dbfs/tmp/chicago_stream_e_checkpoint"

# Clean up and prepare
if os.path.exists(checkpoint_e_path):
    shutil.rmtree(checkpoint_e_path)
if os.path.exists(streaming_e_dir):
    shutil.rmtree(streaming_e_dir)

os.mkdir(streaming_e_dir)
stream_data_e_file = f"{streaming_e_dir}/data.csv"

# Download edge CSV from URL
print(f"Downloading edge CSV from {csv_edge_url}...")
urllib.request.urlretrieve(csv_edge_url, stream_data_e_file)
print(f"Edge CSV downloaded to {stream_data_e_file}")

load_time = time.time()
print(f"\nData setup: {load_time - start_time:.2f}s")

# Load GeoJSON neighborhoods from URL
print(f"Loading GeoJSON from {geojson_url}...")
gdf = gpd.read_file(geojson_url)
print(f"Neighborhoods loaded: {len(gdf)}")

# Broadcast neighborhoods
neighborhood_data = [(row.geometry, row.get('name', f"Area_{i}")) 
                     for i, row in gdf.iterrows()]
neighborhoods_broadcast = spark.sparkContext.broadcast(neighborhood_data)

geojson_time = time.time()
print(f"GeoJSON loading: {geojson_time - load_time:.2f}s")

# Define geohash UDF
def generate_geohash(lat, lon):
    if lat is None or lon is None:
        return None
    return geohash2.encode(lat, lon, precision=GEOHASH_PRECISION)

geohash_udf = udf(generate_geohash, StringType())

# Spatial join UDF
def find_neighborhood(lon, lat):
    if lon is None or lat is None:
        return None
    point = Point(lon, lat)
    for geom, name in neighborhoods_broadcast.value:
        if geom.contains(point):
            return name
    return None

find_neighborhood_udf = udf(find_neighborhood, StringType())

In [0]:

start_time = time.time()

print("\n=== STREAMING PATH (Full Data) ===")

# Spark needs paths without /dbfs prefix
spark_streaming_dir = streaming_dir.replace('/dbfs', '')
spark_streaming_e_dir = streaming_e_dir.replace('/dbfs', '')
spark_checkpoint_path = checkpoint_path.replace('/dbfs', '')
spark_checkpoint_e_path = checkpoint_e_path.replace('/dbfs', '')

'''
# Read CSV as STREAM for full data
stream_df = spark.readStream \
    .option("header", "true") \
    .schema(schema) \
    .csv(spark_streaming_dir)

# Add geohash and neighborhood to stream
stream_with_features = stream_df \
    .withColumn("geohash", geohash_udf(col("Latitude"), col("Longitude"))) \
    .withColumn("neighborhood", find_neighborhood_udf(col("Longitude"), col("Latitude")))

# Tumbling window aggregation (NO WATERMARK)
full_data_stream = stream_with_features \
    .groupBy(
        window("ReadingDateTimeUTC", WINDOW_DURATION),
        "neighborhood"
    ).agg(
        F.avg("PM25").alias("avg_PM25_full"),
        F.avg("Temperature").alias("avg_Temperature_full"),
        F.avg("Humidity").alias("avg_Humidity_full"),
        F.count("*").alias("count_full")
    )

# Write streaming results to memory
full_query = full_data_stream.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("full_data_results") \
    .option("checkpointLocation", f"{spark_checkpoint_path}/full") \
    .start()

print(f"Streaming query started: {full_query.id}")
print("Waiting for streaming data to process...")

# Wait for streaming query to process data (with timeout)
max_wait = 10  # max seconds
wait_interval = 1
elapsed = 0
while elapsed < max_wait:
    time.sleep(wait_interval)
    elapsed += wait_interval
    
    # Check if data has been processed
    try:
        row_count = spark.sql("SELECT COUNT(*) as cnt FROM full_data_results").collect()[0]['cnt']
        if row_count > 0:
            print(f"Streaming query processed {row_count} aggregated rows after {elapsed}s")
            break
        else:
            #print(f"  Waiting... ({elapsed}s elapsed, {row_count} rows so far)")
            pass
    except:
        print(f"  Waiting... ({elapsed}s elapsed)")

if elapsed >= max_wait:
    print(f"Warning: Streaming query did not process data within {max_wait}s")

stream_time = time.time()
print(f"Streaming setup: {stream_time - geojson_time:.2f}s")

print("\n=== BATCH PATH (Sampled Data) ===")

# Read same data as BATCH for sampling
batch_df = spark.read.format("csv").option("header", "true").schema(schema).load(f"{spark_streaming_dir}/data.csv")

print(f"Total records: {batch_df.count()}")

# Add geohash
batch_with_geohash = batch_df.withColumn(
    "geohash", 
    geohash_udf(col("Latitude"), col("Longitude"))
)

print(f"Unique geohashes: {batch_with_geohash.select('geohash').distinct().count()}")

# Get geohash distribution for stratified sampling
geohash_counts = batch_with_geohash.groupBy("geohash").count().collect()
geohash_fractions = {row['geohash']: SAMPLING_FRACTION for row in geohash_counts if row['geohash'] is not None}

print(f"Stratified sampling with {len(geohash_fractions)} geohash strata")

# Stratified sampling by geohash
sampled_data = batch_with_geohash \
    .sampleBy("geohash", fractions=geohash_fractions, seed=42) \
    .withColumn("neighborhood", find_neighborhood_udf(col("Longitude"), col("Latitude"))) \
    .groupBy(
        window("ReadingDateTimeUTC", WINDOW_DURATION),
        "neighborhood"
    ).agg(
        F.avg("PM25").alias("avg_PM25_sampled"),
        F.avg("Temperature").alias("avg_Temperature_sampled"),
        F.avg("Humidity").alias("avg_Humidity_sampled"),
        F.count("*").alias("count_sampled")
    )

sampled_time = time.time()
print(f"Sampled data processing: {sampled_time - stream_time:.2f}s")

processing_time = time.time()
e2e = processing_time - start_time
print(f"\nend-to-end latency (e2e): {e2e:.2f}s")
'''
# --- EDGE DATA Start --- #
# stream and process edge data for comparison
start_time_edge = time.time()

stream_edge_df = spark.readStream \
    .option("header", "true") \
    .schema(schemaEdge) \
    .csv(spark_streaming_e_dir)

full_edge_stream = stream_edge_df \
    .groupBy(
        window("ReadingDateTimeUTC", WINDOW_DURATION),
        "neighborhood"
    ).agg(
        F.avg("PM25").alias("avg_PM25_edge"),
        F.avg("Temperature").alias("avg_Temperature_edge"),
        F.avg("Humidity").alias("avg_Humidity_edge"),
        F.count("*").alias("count_edge")
    )

# Write streaming results to memory
full_edge_query = full_edge_stream.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("full_data_edge_results") \
    .option("checkpointLocation", f"{spark_checkpoint_e_path}/full") \
    .start()

print(f"Streaming query started: {full_edge_query.id}")
print("Waiting for streaming data to process...")

start_time_edge = time.time()
# Wait for streaming query to process data (with timeout)
max_wait = 10 # max seconds
wait_interval = 1
elapsed = 0
while elapsed < max_wait:
    time.sleep(wait_interval)
    elapsed += wait_interval
    
    # Check if data has been processed
    try:
        row_count = spark.sql("SELECT COUNT(*) as cnt FROM full_data_edge_results").collect()[0]['cnt']
        if row_count > 0:
            print(f"Streaming query processed {row_count} aggregated rows after {elapsed}s")
            break
        else:
            #print(f"  Waiting... ({elapsed}s elapsed, {row_count} rows so far)")
            pass
    except:
        print(f"  Waiting... ({elapsed}s elapsed)")

if elapsed >= max_wait:
    print(f"Warning: Streaming query did not process data within {max_wait}s")

processing_time = time.time()
e2e_edge = processing_time - start_time_edge
print(f"\nend-to-end latency (e2e): {e2e_edge:.2f}s")

# --- EDGE DATA End --- #
'''
print("\n=== COMPARISON ===")

# Get streaming results from memory
full_data = spark.sql("SELECT * FROM full_data_results")
print(f"Full data aggregated rows: {full_data.count()}")
print(f"Sampled data aggregated rows: {sampled_data.count()}")
full_edge_data = spark.sql("SELECT * FROM full_data_edge_results")

# Join full and sampled results
comparison = full_data.join(
    sampled_data,
    ["window", "neighborhood"],
    "inner"
).join(
    full_edge_data,
    ["window", "neighborhood"],
    "inner"
).select(
    col("window"),
    col("neighborhood"),
    col("avg_PM25_full"),
    col("avg_PM25_sampled"),
    col("avg_PM25_edge"),
    col("avg_Temperature_full"),
    col("avg_Temperature_sampled"),
    col("avg_Temperature_edge"),
    col("avg_Humidity_full"),
    col("avg_Humidity_sampled"),
    col("avg_Humidity_edge"),
    col("count_full"),
    col("count_sampled"),
    col("count_edge"),
    # Calculate errors
    F.pow(col("avg_PM25_full") - col("avg_PM25_sampled"), 2).alias("pm25_squared_error"),
    F.pow(col("avg_PM25_full") - col("avg_PM25_edge"), 2).alias("pm25_squared_error_edge"),
    (F.abs(col("avg_PM25_full") - col("avg_PM25_sampled")) / F.abs(col("avg_PM25_full")) * 100).alias("pm25_ape"),
    (F.abs(col("avg_PM25_full") - col("avg_PM25_edge")) / F.abs(col("avg_PM25_full")) * 100).alias("pm25_ape_edge"),
    F.pow(col("avg_Temperature_full") - col("avg_Temperature_sampled"), 2).alias("temp_squared_error"),
    F.pow(col("avg_Temperature_full") - col("avg_Temperature_edge"), 2).alias("temp_squared_error_edge"),
    (F.abs(col("avg_Temperature_full") - col("avg_Temperature_sampled")) / F.abs(col("avg_Temperature_full")) * 100).alias("temp_ape"),
    (F.abs(col("avg_Temperature_full") - col("avg_Temperature_edge")) / F.abs(col("avg_Temperature_full")) * 100).alias("temp_ape_edge"),
    F.pow(col("avg_Humidity_full") - col("avg_Humidity_sampled"), 2).alias("humidity_squared_error"),
    F.pow(col("avg_Humidity_full") - col("avg_Humidity_edge"), 2).alias("humidity_squared_error_edge"),
    (F.abs(col("avg_Humidity_full") - col("avg_Humidity_sampled")) / F.abs(col("avg_Humidity_full")) * 100).alias("humidity_ape"),
    (F.abs(col("avg_Humidity_full") - col("avg_Humidity_edge")) / F.abs(col("avg_Humidity_full")) * 100).alias("humidity_ape_edge")
)

processing_time = time.time()
print(f"Comparison join: {processing_time - sampled_time:.2f}s")
print(f"Comparison result rows: {comparison.count()}")
print(f"\nTotal end-to-end latency: cloud-only {e2e:.2f}s; edge: {e2e_edge:.2f}s")
print("\nDisplaying comparison results...")

# Display comparison
display(comparison.orderBy("window", "neighborhood"))

# Group by neighborhood and calculate average PM2.5
print("\n=== AVERAGE PM2.5 BY NEIGHBORHOOD ===")
neighborhood_avg = comparison.groupBy("neighborhood").agg(
    F.avg("avg_PM25_full").alias("avg_PM25_full"),
    F.avg("avg_PM25_sampled").alias("avg_PM25_sampled"),
    F.avg("avg_PM25_edge").alias("avg_PM25_edge"),
    F.sum("count_full").alias("total_records"),
    F.count("*").alias("num_windows")
).orderBy(F.desc("total_records"))

print("\nAverage PM2.5 per Neighborhood (Full vs Sampled vs Edge):")
display(neighborhood_avg)

# Stop streaming query
full_query.stop()
full_edge_query.stop()
print("\nStreaming query stopped.")
'''